# 02 · Swin-T detector: detection-only fine-tuning at 512×768

Fine-tunes the Swin-T Faster R-CNN for **detection only**, at a higher resolution of 512×768. The ReID head is skipped and all annotated people are used, including those with unknown identity. The result is `swin_detector.pt`, the detector shared by the three Swin systems in `main.ipynb` (top row of diagrams 2–4). After this step it is frozen.

- **Warm start:** `joint_swin_t_triplet.pt`, the joint Swin-T model trained by `01_joint_person_search` with `CONFIG = "joint_swin_t_triplet"`.
- **Training:** random batches of 16 frames, AdamW with backbone lr 1e-5 and FPN/RPN/head lr 1e-4, weight decay 1e-4, AMP.
- **Selection:** validation mean(AP50, AP75) on the same 10% frame split (seed 0). The warm-start model is evaluated first and kept unless an epoch improves on it.
- **Epochs:** the published detector is **epoch 9** of a run configured for 50 epochs that stopped after epoch 10 (≈ 3.9 h on a T4). `NUM_EPOCHS = 10` reproduces it within one Kaggle session.

**Run time (Kaggle T4):** ≈ 4 h.

### How to run
1. Create a Kaggle notebook from this file (*File → Import Notebook*), then attach the inputs listed below with *Add Input*.
2. Settings: **Accelerator → GPU T4 ×1**. Internet is only needed if you set `USE_WANDB = True`.
3. **Run All.** The inputs are found automatically wherever Kaggle mounts them under `/kaggle/input`.

Weights & Biases logging is **off** by default (`USE_WANDB = False`), so no account or secret is needed. To log curves and upload checkpoints, set `USE_WANDB = True` and store your key as the Kaggle secret / environment variable `WANDB_API_KEY`.

**Inputs**

| Input | Kaggle source | Used for |
|---|---|---|
| PRW dataset | `edoardomerli/prw-person-re-identification-in-the-wild` | frames, annotations, queries |
| Code | `saadqasimabbasi/prw-person-search-code` | the project modules in `src/` (and the DINOv3 code snapshot) |
| Warm start | output of `01_joint_person_search` (`joint_swin_t_triplet.pt`) | initial detector weights |

## 1. Setup: locate the Kaggle inputs
This cell scans `/kaggle/input` once and finds the project code (the `prw-person-search-code` dataset), the PRW dataset and the checkpoint files this notebook needs. It fails with a clear message if one is missing, then adds the code folder to `sys.path`. Outputs go to `/kaggle/working`. Set the environment variables `PRW_INPUT_ROOT` / `PRW_WORKING` to run outside Kaggle.

> **Run time:** a few seconds.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path(os.environ.get("PRW_INPUT_ROOT", "/kaggle/input"))
WORKING = Path(os.environ.get("PRW_WORKING", "/kaggle/working"))
USE_WANDB = False  # True: log curves and upload checkpoints to W&B (needs WANDB_API_KEY)
os.environ["WANDB_MODE"] = "online" if USE_WANDB else "disabled"
os.environ["WANDB_DIR"] = str(WORKING)

PROJECT_MODULES = {"model.py", "training.py", "prw_dataset.py", "prw_evaluation.py"}
REQUIRED_FILES = {"joint_swin_t_triplet.pt"}
SKIPPED_FOLDERS = {"frames", "annotations", "query_box", ".git"}  # thousands of images, nothing to find


def scan_inputs():
    """Walk the input folder once and return the code, DINOv3, PRW and checkpoint locations."""
    code_dir = dinov3_dir = prw_root = None
    found = {}
    for directory, subfolders, files in os.walk(INPUT_ROOT, followlinks=True):
        directory = Path(directory)
        if prw_root is None and "query_info.txt" in files and "frames" in subfolders:
            prw_root = directory
        subfolders[:] = [name for name in subfolders if name not in SKIPPED_FOLDERS]
        if code_dir is None and PROJECT_MODULES.issubset(files):
            code_dir = directory
        if dinov3_dir is None and "hubconf.py" in files and directory.name.startswith("dinov3"):
            dinov3_dir = directory
        for name in REQUIRED_FILES.intersection(files):
            found.setdefault(name, directory / name)
    return code_dir, dinov3_dir, prw_root, found


CODE_DIR, DINOV3_DIR, PRW_ROOT, INPUT_FILES = scan_inputs()
missing = [label for label, value in [("code dataset (project .py files)", CODE_DIR),
                                      ("PRW dataset", PRW_ROOT)] if value is None]
missing += [f"input file {name}" for name in sorted(REQUIRED_FILES - set(INPUT_FILES))]
if missing:
    raise FileNotFoundError("Missing inputs (see 'How to run'): " + ", ".join(missing))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
WORKING.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Code:  ", CODE_DIR)
print("PRW:   ", PRW_ROOT)
for name, path in sorted(INPUT_FILES.items()):
    print("Input: ", name, "->", path)
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU: enable one in the settings)")

## 2. Configuration, warm start and data
The warm-start checkpoint provides the architecture (`model_config`), the weights and the split seed. The model is rebuilt with `detection_only=True`, which disables the ReID branch. The frames are re-loaded at 512×768 without query crops. `evaluate_detector` (`src/detection_training.py`) then measures the starting point at the new resolution.

> **Run time (Kaggle T4):** ≈ 2 min (estimate; includes one validation pass).

In [ ]:
import json
import random

from torch.utils.data import DataLoader

from detection_training import detection_epoch, evaluate_detector
from model import PersonSearchModel
from prw_dataset import PRWPersonSearchDataset, PRWSubset, prw_collate_fn
from training import make_grad_scaler, start_run

SOURCE_CHECKPOINT = INPUT_FILES["joint_swin_t_triplet.pt"]
IMAGE_SIZE = (512, 768)
BATCH_SIZE = 16
NUM_EPOCHS = 10
NUM_WORKERS = 2
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4

state = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
config = dict(state.get("detector_config", state.get("model_config", {})))
if config.get("backbone_name") != "swin_t":
    raise ValueError("Expected the Swin-T checkpoint from 01_joint_person_search")
SEED = int(state["run_config"]["seed"])
VAL_FRACTION = float(state["run_config"].get("val_fraction", .10))
random.seed(SEED)
torch.manual_seed(SEED)
config.update(pretrained=False, detection_only=True)
model = PersonSearchModel(**config).to(device)
model.load_state_dict(state.get("detector_state_dict", state.get("model_state_dict")), strict=True)
del state

full = PRWPersonSearchDataset(PRW_ROOT, split="train", image_size=IMAGE_SIZE, include_query_crops=False)
indices = torch.randperm(len(full), generator=torch.Generator().manual_seed(SEED)).tolist()
cut = round(len(indices) * VAL_FRACTION)
train_data, val_data = PRWSubset(full, indices[cut:]), PRWSubset(full, indices[:cut])


def make_loader(data, shuffle=False):
    return DataLoader(data, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      collate_fn=prw_collate_fn, pin_memory=device.type == "cuda")


train_loader, val_loader = make_loader(train_data, shuffle=True), make_loader(val_data)
print("train/validation frames:", len(train_data), len(val_data), "seed:", SEED)
run_config = dict(experiment="swin_detection_only_512x768", seed=SEED, val_fraction=VAL_FRACTION,
                  image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS,
                  source_checkpoint=str(SOURCE_CHECKPOINT), model_config=config,
                  backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, checkpoint_selection="mean_AP50_AP75")
run = start_run(run_config, name="swin-detector-only-512x768")
baseline = evaluate_detector(model, val_loader, device)
run.log({"epoch": 0, **{f"val/{k}": v for k, v in baseline.items()}})
print("validation before fine-tuning:", baseline)

## 3. Fine-tune and checkpoint
The backbone and the detection layers get separate learning rates. After every epoch the detector is evaluated on the validation frames. `best.pt` keeps the epoch with the highest mean(AP50, AP75), starting from the warm-start score. When the loop ends, `best.pt` is copied to `/kaggle/working/swin_detector.pt`.

> **Run time (Kaggle T4):** ≈ 23 min per epoch, ≈ 3.9 h for 10 epochs.

In [ ]:
import shutil

import wandb

optimizer = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": BACKBONE_LR},
    {"params": [p for name, p in model.named_parameters()
                if p.requires_grad and not name.startswith("backbone.")], "lr": HEAD_LR},
], weight_decay=1e-4)
scaler = make_grad_scaler(device)
checkpoint_dir = WORKING / "checkpoints" / "swin_detector"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best = baseline["mean_AP50_AP75"]


def save_checkpoint(epoch, metrics, path):
    torch.save(dict(epoch=epoch, model_config=config, model_state_dict=model.state_dict(),
                    optimizer_state_dict=optimizer.state_dict(), scaler_state_dict=scaler.state_dict(),
                    run_config=run_config, detector_image_size=IMAGE_SIZE,
                    best_detection_ap=best, validation_metrics=metrics), path)


save_checkpoint(0, baseline, checkpoint_dir / "best.pt")  # kept if no epoch improves validation
for epoch in range(1, NUM_EPOCHS + 1):
    losses = detection_epoch(model, train_loader, optimizer, scaler, device)
    metrics = evaluate_detector(model, val_loader, device)
    run.log({"epoch": epoch, **{f"train/{k}": v for k, v in losses.items()},
             **{f"val/{k}": v for k, v in metrics.items()}})
    if metrics["mean_AP50_AP75"] > best:
        best = metrics["mean_AP50_AP75"]
        save_checkpoint(epoch, metrics, checkpoint_dir / "best.pt")
    save_checkpoint(epoch, metrics, checkpoint_dir / "latest.pt")
    print(f"epoch {epoch}: {metrics}; best={best:.4f}", flush=True)

shutil.copy2(checkpoint_dir / "best.pt", WORKING / "swin_detector.pt")
if USE_WANDB:
    artifact = wandb.Artifact("swin-detector-512x768-best", type="model", metadata=run_config)
    artifact.add_file(str(WORKING / "swin_detector.pt"))
    run.log_artifact(artifact)
print("saved", WORKING / "swin_detector.pt")

## 4. Detection test
The selected detector is evaluated once on the official test frames: AP50, AP75 and precision/recall at confidence 0.5. This measures detection only; ReID is evaluated in `main.ipynb`.

> **Run time (Kaggle T4):** ≈ 10–15 min (estimate).

In [ ]:
best_state = torch.load(checkpoint_dir / "best.pt", map_location="cpu", weights_only=False)
model.load_state_dict(best_state["model_state_dict"], strict=True)
selected_epoch = best_state["epoch"]
del best_state
test_data = PRWPersonSearchDataset(PRW_ROOT, split="test", image_size=IMAGE_SIZE, include_query_crops=False)
test_metrics = evaluate_detector(model, make_loader(test_data), device)
(checkpoint_dir / "test_detection.json").write_text(json.dumps(dict(checkpoint_epoch=selected_epoch,
                                                                    metrics=test_metrics), indent=2))
run.summary.update({f"test_detection/{k}": v for k, v in test_metrics.items()})
run.finish()
print("selected epoch:", selected_epoch)
print("test detection:", test_metrics)